In [77]:
import pandas as pd
import mysql.connector
import seaborn as sns
import matplotlib.pyplot as plt
from config.db_config import DB_CONFIG
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
import numpy as np


In [78]:
conn = mysql.connector.connect(**DB_CONFIG, database="salary_prediction")
cursor = conn.cursor()

df_developers = pd.read_sql("SELECT * FROM developers", conn)
print(df_developers.columns)


languages_sql = """
SELECT d.respondent_id, d.converted_comp_yearly, l.language_name
FROM developers d 
    INNER JOIN developer_languages dl ON d.respondent_id = dl.respondent_id 
    INNER JOIN languages l ON l.language_id = dl.language_id
ORDER BY d.respondent_id;
"""
df_language = pd.read_sql(languages_sql, conn)
print(df_language.columns)


Index(['respondent_id', 'country', 'years_code_pro', 'ed_level', 'dev_type',
       'employment', 'converted_comp_yearly'],
      dtype='object')
Index(['respondent_id', 'converted_comp_yearly', 'language_name'], dtype='object')


In [79]:
lang_pivot = df_language.pivot_table(index="respondent_id", columns="language_name", aggfunc="size", fill_value=0)
combined_df = pd.merge(df_developers, lang_pivot, on="respondent_id", how="inner")
combined_df.head()

formatted_df = pd.get_dummies(combined_df, columns=["country", "dev_type", "employment"])
formatted_df.columns


Index(['respondent_id', 'years_code_pro', 'ed_level', 'converted_comp_yearly',
       'Ada', 'Apex', 'Assembly', 'Bash/Shell (all shells)', 'C', 'C#',
       ...
       'dev_type_Product manager', 'dev_type_Project manager',
       'dev_type_Research & Development role', 'dev_type_Scientist',
       'dev_type_Security professional',
       'dev_type_Senior Executive (C-Suite, VP, etc.)',
       'dev_type_System administrator', 'employment_Employed_full_time',
       'employment_Employed_part_time', 'employment_Self_employed'],
      dtype='object', length=253)

In [80]:


y = formatted_df["converted_comp_yearly"].copy()
X = formatted_df.drop(["respondent_id", "converted_comp_yearly"], axis=1).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



## Linear Regression

In [81]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)

print(f"R^2: {model.score(X_test, y_test)}")
print(f"RMSE: {rmse}")

R^2: 0.597162100043481
RMSE: 32167.496344378516


## Random Forest

In [82]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_rmse = root_mean_squared_error(y_test, rf_pred)

print(f"R^2: {rf_model.score(X_test, y_test)}")
print(f"RMSE: {rf_rmse}")

R^2: 0.6045801606062511
RMSE: 31869.9459057988


Based on the result and EDA, I decided:

- Removed `Employed_part_time`: lower salaries likely reflect fewer working hours, not the features being modeled
- Filtered `country` to top 25 (covering 80% of data) to reduce noise from underrepresented countries
- Filtered `dev_type` to top 15 (covering 91% of data) for the same reason
- Applied log transformation to salary to address right-skewed distribution
- Testing models with and without `ed_level` since correlation with salary was only 0.094
- Applied log transformation to `years_code_pro` to address right-skewed distribution (used `log1p` to handle values less than 1)

In [83]:
# Remove Employed part time
experiment_df = combined_df[combined_df["employment"] != "Employed_part_time"]
experiment_df.head()

# Fileter top 25 countries
df_top25_countries = experiment_df["country"].value_counts().head(25)

top25 = df_top25_countries.index
experiment_df = experiment_df[experiment_df["country"].isin(top25)]

# Filter top 15 dev types
df_top_15_dev_type = experiment_df["dev_type"].value_counts().head(15)

top15_dev_type = df_top_15_dev_type.index
experiment_df = experiment_df[experiment_df["dev_type"].isin(top15_dev_type)]


# # Apply log transformation on converted_comp_yearly
# experiment_df["converted_comp_yearly"] = np.log(experiment_df["converted_comp_yearly"])

# # Apply log transformation(log1p) on years_code_pro
# experiment_df["years_code_pro"] = np.log1p(experiment_df["years_code_pro"])

# fig, ax = plt.subplots(nrows=1, ncols=2)

# sns.histplot(experiment_df["years_code_pro"], ax=ax[0])
# sns.histplot(experiment_df["converted_comp_yearly"], ax=ax[1])
# plt.show()
# experiment_df["converted_comp_yearly"].describe()

It seems converted_comp_yearly contains 0, which means log(1) = 0. The original data contains the people who earn 1 dollar, so I decided to remove them.

In [84]:
# # Revert log transformation
# experiment_df["converted_comp_yearly"] = np.exp(experiment_df["converted_comp_yearly"])

# # Remove extremely low salaries (under $1000)
# experiment_df = experiment_df[experiment_df["converted_comp_yearly"] > 1000]

# # Re-apply log transformation
# experiment_df["converted_comp_yearly"] = np.log(experiment_df["converted_comp_yearly"])

# # Verify
# fig, ax = plt.subplots(nrows=1, ncols=2)
# sns.histplot(experiment_df["years_code_pro"], ax=ax[0])
# sns.histplot(experiment_df["converted_comp_yearly"], ax=ax[1])
# plt.show()
# experiment_df["converted_comp_yearly"].describe()

In [85]:
# one-hot encoding
experiment_df = pd.get_dummies(experiment_df, columns=["country", "dev_type", "employment"])
experiment_df.head()
with pd.option_context('display.max_columns', None):
    display(experiment_df.columns)
y = experiment_df["converted_comp_yearly"].copy()
X = experiment_df.drop(["respondent_id", "converted_comp_yearly"], axis=1).copy()



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Index(['respondent_id', 'years_code_pro', 'ed_level', 'converted_comp_yearly',
       'Ada', 'Apex', 'Assembly', 'Bash/Shell (all shells)', 'C', 'C#', 'C++',
       'Clojure', 'Cobol', 'Crystal', 'Dart', 'Delphi', 'Elixir', 'Erlang',
       'F#', 'Fortran', 'GDScript', 'Go', 'Groovy', 'HTML/CSS', 'Haskell',
       'Java', 'JavaScript', 'Julia', 'Kotlin', 'Lisp', 'Lua', 'MATLAB',
       'MicroPython', 'Nim', 'OCaml', 'Objective-C', 'PHP', 'Perl',
       'PowerShell', 'Prolog', 'Python', 'R', 'Ruby', 'Rust', 'SQL', 'Scala',
       'Solidity', 'Swift', 'TypeScript', 'VBA', 'Visual Basic (.Net)',
       'Zephyr', 'Zig', 'country_Australia', 'country_Austria',
       'country_Belgium', 'country_Brazil', 'country_Canada',
       'country_Czech Republic', 'country_Denmark', 'country_France',
       'country_Germany', 'country_Greece', 'country_India', 'country_Israel',
       'country_Italy', 'country_Netherlands', 'country_Norway',
       'country_Poland', 'country_Portugal', 'country_Russia

## Liearn Regression 2nd

In [86]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)

print(f"R^2: {model.score(X_test, y_test)}")
print(f"RMSE: {rmse}")

R^2: 0.6200339213354646
RMSE: 32018.998627549063


## Random Forest 2nd

In [87]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_rmse = root_mean_squared_error(y_test, rf_pred)

print(f"R^2: {rf_model.score(X_test, y_test)}")
print(f"RMSE: {rf_rmse}")

R^2: 0.6378192800010789
RMSE: 31260.649420621477


## Result

### Linear:
No Log No FIlter: 
- R^2: 0.597162100043481 
- RMSE: 32167.496344378516

No Log Yes FIlter:
- R^2: 0.6200339213354646 
- RMSE: 32018.998627549063

Yes Log, No Filter:
- R^2: 0.5344096228716414
- RMSE: 0.6838642738599746

### RF:
No Log No FIlter: 
- R^2: 0.6045801606062511 
- RMSE: 31869.9459057988

No Log, Yes FIlter:
- R^2: 0.6378192800010789 
- RMSE: 31260.649420621477

Yes Log, No Filter:
- R^2: 0.49371987057397315
- RMSE: 0.713121219597329

## Gradient Boosting

In [88]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)
gb_rmse = root_mean_squared_error(y_test, gb_pred)

print(f"R^2: {gb_model.score(X_test, y_test)}")
print(f"RMSE: {gb_rmse}")

R^2: 0.6350861059462704
RMSE: 31378.380988300596


## XGBoost

In [91]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_rmse = root_mean_squared_error(y_test, xgb_pred)

print(f"R^2: {xgb_model.score(X_test, y_test)}")
print(f"RMSE: {xgb_rmse}")

importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(20)

R^2: 0.6498772762552684
RMSE: 30735.86792232954


country_United States of America                                0.298920
country_India                                                   0.076781
country_Ukraine                                                 0.063894
country_Brazil                                                  0.048074
country_Switzerland                                             0.042290
country_Italy                                                   0.036197
country_Russian Federation                                      0.029106
country_Israel                                                  0.024371
country_Turkey                                                  0.021185
country_United Kingdom of Great Britain and Northern Ireland    0.017591
country_Australia                                               0.017359
dev_type_Academic researcher                                    0.017284
country_Canada                                                  0.016612
country_Denmark                                    

In [92]:
important_features = importances[importances > 0.001].index
X_train_reduced = X_train[important_features]
X_test_reduced = X_test[important_features]

In [94]:
xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train_reduced, y_train)

xgb_pred = xgb_model.predict(X_test_reduced)
xgb_rmse = root_mean_squared_error(y_test, xgb_pred)

print(f"R^2: {xgb_model.score(X_test_reduced, y_test)}")
print(f"RMSE: {xgb_rmse}")

R^2: 0.6499260481750715
RMSE: 30733.72710228061


In [96]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9, 11],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

grid_search = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best R^2: {grid_search.best_score_}")

Best params: {'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 1000, 'subsample': 0.7}
Best R^2: 0.6602994278584626


In [97]:
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)
best_rmse = root_mean_squared_error(y_test, best_pred)

print(f"R^2: {best_model.score(X_test, y_test)}")
print(f"RMSE: {best_rmse}")

R^2: 0.6685476858328413
RMSE: 29905.141376646596
